In [ ]:
import os
import glob
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs

folder_path = './cyp_toxicogenomics/zinc/SMILES_B'
smi_files = glob.glob(os.path.join(folder_path, '*.smi'))

# List of substrate/toxicant SMILES
substrate_smiles_list = [
    "c1ccccc1",
    "c1ccc(Cc2ccccc2)cc1",
    "O=c1ccn(-c2ccccc2)c2nc(N3CC4CC4C3)ccc12",
    "c1cncc(C2CCCN2)c1",
    "c1ccc2c(c1)Cc1ccccc1-2",
    "c1ccc(-c2ccccc2)cc1",
    "c1ccc2cnncc2c1",
    "O=C1CCc2c1c(=O)oc1c3c(ccc21)OC1OC=CC31",
    "c1cnc2ccc3[nH]cnc3c2c1",
    "O=C(c1ccccc1)c1ccccc1",
    "c1ccc2c(c1)cc1ccc3cccc4ccc2c1c34"
]

# Convert each substrate SMILES to a fingerprint
toxic_fps = []
for smi in substrate_smiles_list:
    mol = Chem.MolFromSmiles(smi)
    if mol:
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)
        toxic_fps.append(fp)

# Load SMILES and ZINC IDs from all files
db_entries = []  # List of (smiles, zinc_id)
for file in smi_files:
    with open(file, 'r') as f:
        next(f)  # Skip header
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                smi, zinc_id = parts[0], parts[1]
                db_entries.append((smi, zinc_id))

# Filter valid molecules and compute fingerprints
db_mols = []
valid_entries = []
for smi, zinc_id in db_entries:
    mol = Chem.MolFromSmiles(smi)
    if mol:
        db_mols.append(mol)
        valid_entries.append((smi, zinc_id))

db_fps = [AllChem.GetMorganFingerprintAsBitVect(mol, 2, 2048) for mol in db_mols]

# Compute max similarity to the toxicant
threshold = 0.85
toxic_like = []
for (smi, zinc_id), db_fp in zip(valid_entries, db_fps):
    max_sim = max(DataStructs.TanimotoSimilarity(db_fp, tox_fp) for tox_fp in toxic_fps)
    if max_sim >= threshold:
        toxic_like.append((smi, zinc_id, max_sim))

output_file = 'substrate_toxic_dataset.txt'
with open(output_file, 'w') as out_f:
    out_f.write("SMILES\tZINC_ID\tSimilarity\n")
    for smi, zinc_id, sim in toxic_like:
        out_f.write(f"{smi}\t{zinc_id}\t{sim:.3f}\n")

print(f"Found {len(toxic_like)} substrate toxic-like molecules. Saved to {output_file}")


[23:48:35] DEPRECATION WARNING: please use MorganGenerator
[23:48:35] DEPRECATION WARNING: please use MorganGenerator
[23:48:35] DEPRECATION WARNING: please use MorganGenerator
[23:48:35] DEPRECATION WARNING: please use MorganGenerator
[23:48:35] DEPRECATION WARNING: please use MorganGenerator
[23:48:35] DEPRECATION WARNING: please use MorganGenerator
[23:48:35] DEPRECATION WARNING: please use MorganGenerator
[23:48:35] DEPRECATION WARNING: please use MorganGenerator
[23:48:35] DEPRECATION WARNING: please use MorganGenerator
[23:48:35] DEPRECATION WARNING: please use MorganGenerator
[23:48:35] DEPRECATION WARNING: please use MorganGenerator
